# M2 Notebook 35 — Machine Learning Capstone

**Status:** Runnable first edition

## Objective

Build a governed end-to-end ML decision system.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_ml import LogisticRegressionGD,StandardScaler,accuracy_score,brier_score,dataset_signature,select_capacity_constrained,subgroup_summary,train_test_split,validate_pipeline_inputs,population_stability_index


In [ ]:
rng=np.random.default_rng(35); n=4000
X=pd.DataFrame({'service_gap':rng.normal(size=n),'distance':rng.lognormal(.2,.5,n),'staffing':rng.normal(size=n),'demand':rng.normal(size=n),'seasonality':rng.normal(size=n)})
group=np.where(rng.random(n)>.5,'Urban','Rural'); logit=-1+1.3*X.service_gap-.5*X.staffing+.6*X.demand+.2*(group=='Rural'); p=1/(1+np.exp(-logit)); y=rng.binomial(1,p)
{'checks':validate_pipeline_inputs(X.to_numpy(),y),'signature':dataset_signature(X.to_numpy(),y)}


In [ ]:
Xtr,Xte,ytr,yte=train_test_split(X.to_numpy(),y,test_size=.3,seed=35); _,_,_,gte=train_test_split(X.to_numpy(),group,test_size=.3,seed=35)
sc=StandardScaler(); Xtr_s=sc.fit_transform(Xtr); Xte_s=sc.transform(Xte); model=LogisticRegressionGD(.08,4000,l2=.001).fit(Xtr_s,ytr); prob=model.predict_proba(Xte_s)[:,1]; pred=(prob>=.5).astype(int)
{'accuracy':accuracy_score(yte,pred),'brier':brier_score(yte,prob)}


In [ ]:
decision,value=select_capacity_constrained(prob,250,12,2)
{'selected':int(decision.sum()),'mean_probability_selected':float(prob[decision==1].mean()),'mean_value_selected':float(value[decision==1].mean())}


In [ ]:
subgroup_summary(yte,decision,gte)


In [ ]:
{'service_gap_PSI':population_stability_index(Xtr[:,0],Xte[:,0]+.5,10)}


## Final M2 insight

Machine learning becomes operational when prediction, evaluation, deployment, monitoring, and governance are integrated.